# Paso 3: pseudoetiquetar el audio no pareado con el ASR semilla

Transcribe los 740 clips de `no_pareado_v1` guardando el **log-prob promedio por token** de cada clip, y los filtra:

| # | filtro | descarta si |
|---|---|---|
| 1 | vacío | el ASR no produjo texto |
| 2 | repeticiones | bucle de Whisper: llegó al tope de tokens, *compression ratio* > 2.4 (criterio del propio Whisper), una palabra 4+ veces seguidas o una frase de 2–4 palabras 3+ veces |
| 3 | caracteres por segundo | fuera del rango p0.5–p99.5 del audio real pareado (dataset_v1.0) |
| 4 | confianza | log-prob promedio bajo el umbral, o fuera del X % con más confianza |

**El control set mide la calidad de las pseudoetiquetas.** Los 259 clips que el transcriptor entregó se transcriben
igual que el resto y se comparan con su transcripción humana:
- WER/CER del ASR semilla en audio no pareado (con IC 95 %), por dominio;
- **WER de los clips que pasarían los filtros**: es el error esperado de las etiquetas que entran a `metadata_pseudo.csv`;
- la curva *WER vs. % de clips que se quedan* para elegir X (o el umbral) con datos.

Los clips del control **nunca** entran a `metadata_pseudo.csv`: son la evaluación.

Validado localmente: el filtro de repeticiones no marca ninguna de las 3,586 transcripciones humanas del proyecto
(la reduplicación real llega a 3 veces la misma palabra), y el rango de caracteres por segundo se eligió contra el control humano: con p1–p99
se descartaba el 6.6 % de etiquetas humanas correctas (el no pareado es habla más rápida: spots de radio a
~13 caracteres/s frente a ~10 de SK001); con p0.5–p99.5, el 3.5 %.

In [ ]:
#@title 1. Parámetros
NP_ID = ""  #@param {type:"string"}
DRIVE_NP = "/content/drive/MyDrive/TESIS/asr/no_pareado_v1.zip"  #@param {type:"string"}
DATASET_ID = ""  #@param {type:"string"}
DRIVE_DATASET = "/content/drive/MyDrive/TESIS/asr/dataset_v1.0.zip"  #@param {type:"string"}
#@markdown ASR que pseudoetiqueta (su `modelo_final`). Para repetir con un ASR mejor, cambia esto y SALIDA.
ASR_DIR = "/content/drive/MyDrive/TESIS/asr/whisper_small_base"  #@param {type:"string"}
SALIDA = "/content/drive/MyDrive/TESIS/asr/pseudo_v1"  #@param {type:"string"}

#@markdown **Filtro 4 (confianza)**: "top" = quedarse con el TOP_PCT % de mayor confianza; "umbral" = log-prob >= UMBRAL_LOGPROB
MODO_CONFIANZA = "top"  #@param ["top", "umbral"]
TOP_PCT = 70  #@param {type:"number"}
UMBRAL_LOGPROB = -0.5  #@param {type:"number"}
#@markdown Filtros 2 y 3
COMP_MAX = 2.4
REP_1, REP_N = 4, 3
P_BAJO, P_ALTO = 0.5, 99.5   # p1-p99 descartaba el 6.6% de las etiquetas humanas correctas del control
BATCH = 16  #@param {type:"integer"}
MAX_TOKENS = 128
FP16 = True
SEED = 1234

## 2. GPU y Drive

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No hay GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)")
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")

## 3. Librerías

In [ ]:
!pip install -q transformers==5.17.0 accelerate==1.15.0 jiwer==4.0.0 gdown

## 4. Núcleo (probado localmente)

In [ ]:
%%writefile /content/asr_core.py
# ---- nucleo del ASR: se prueba localmente y se copia tal cual al notebook ----
import hashlib, re, unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch

SR = 16000
FUENTE_SK = "menendez_gomez_2025"


# ============================ TEXTO ============================
def normalizar(t):
    """Misma normalizacion que build_dataset_v1.py (text_norm): NFC, minusculas, sin puntuacion.
    Se aplica tambien a lo que predice Whisper, que tiende a poner mayusculas y puntos."""
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = re.sub(r"[¡!¿?.,;:…\"“”«»()\[\]{}—–\-_/\\]", " ", t)
    t = t.replace("'", " ").replace("’", " ")
    return re.sub(r"\s+", " ", t).strip()


def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


# ============================ SPLIT ============================
def hacer_split(meta, lineas_tts, seed, frac_dev, frac_test, frac_otros):
    """Particion fija para todos los experimentos de la tesis.

    - test_otros: grupos de hablante ENTEROS de youtube y workshop (voces que el modelo no oye).
    - dev / test_sk001: clips de SK001 cuyo texto NO aparece en el txt del TTS, para que los
      experimentos con audio sintetico no vean las frases de evaluacion.
    - Un texto de evaluacion nunca queda tambien en train (se saca de train: 'fuera_texto').
    """
    m = meta.sort_values("id").reset_index(drop=True).copy()
    m["split"] = "train"
    rng = np.random.RandomState(seed)

    # 1) hablantes no vistos
    for fuente in sorted(set(m.source) - {FUENTE_SK}):
        d = m[m.source == fuente].groupby("speaker_group").duration_s.sum()
        grupos = sorted(d.index)
        rng.shuffle(grupos)
        acum, elegidos = 0.0, []
        for g in grupos:
            if acum >= frac_otros * d.sum():
                break
            elegidos.append(g)
            acum += d[g]
        m.loc[m.speaker_group.isin(elegidos), "split"] = "test_otros"

    # 2) SK001: textos que no esten contenidos en ninguna linea del TTS
    blob = "\n".join(f" {normalizar(l)} " for l in lineas_tts if l.strip())
    sk = m[m.source == FUENTE_SK]
    elegibles = sorted(t for t in set(sk.text_norm) if f" {t} " not in blob)
    rng.shuffle(elegibles)
    n_test, n_dev = round(frac_test * len(sk)), round(frac_dev * len(sk))
    cuenta = sk.text_norm.value_counts()
    usados = {"test_sk001": 0, "dev": 0}
    for t in elegibles:
        destino = "test_sk001" if usados["test_sk001"] < n_test else "dev" if usados["dev"] < n_dev else None
        if destino is None:
            break
        m.loc[(m.source == FUENTE_SK) & (m.text_norm == t), "split"] = destino
        usados[destino] += int(cuenta[t])

    # 3) sin fuga por texto: evaluacion con texto en el TTS, o train con texto de evaluacion
    evaluacion = m.split != "train"
    en_tts = m.text_norm.map(lambda t: f" {t} " in blob)
    m.loc[evaluacion & en_tts, "split"] = "fuera_tts"
    textos_eval = set(m.loc[m.split.isin(["dev", "test_sk001", "test_otros"]), "text_norm"])
    m.loc[(m.split == "train") & m.text_norm.isin(textos_eval), "split"] = "fuera_texto"
    return m


def resumen_split(m):
    r = m.groupby("split").agg(clips=("id", "size"), minutos=("duration_s", lambda s: round(s.sum() / 60, 1)),
                               hablantes=("speaker_group", "nunique"))
    return r.reindex([s for s in ["train", "dev", "test_sk001", "test_otros", "fuera_tts", "fuera_texto"]
                      if s in r.index])


# ============================ DATOS PARA WHISPER ============================
class ClipsASR(torch.utils.data.Dataset):
    def __init__(self, df, raiz, processor):
        self.filas = list(df[["file_name", "text_norm"]].itertuples(index=False))
        self.raiz, self.p = Path(raiz), processor

    def __len__(self):
        return len(self.filas)

    def __getitem__(self, i):
        archivo, texto = self.filas[i]
        y, sr = sf.read(self.raiz / archivo, dtype="float32")
        assert sr == SR, f"{archivo}: {sr} Hz"
        return {"input_features": self.p.feature_extractor(y, sampling_rate=SR).input_features[0],
                "labels": self.p.tokenizer(texto).input_ids}


@dataclass
class Collator:
    processor: object
    decoder_start_token_id: int

    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        lab = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab["attention_mask"].ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all():   # el modelo lo agrega solo
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


# ============================ METRICAS ============================
def wer_cer(refs, hyps):
    import jiwer
    pares = [(r, h) for r, h in zip(refs, hyps) if r]            # jiwer no acepta referencias vacias
    r, h = [p[0] for p in pares], [p[1] for p in pares]
    return 100 * jiwer.wer(r, h), 100 * jiwer.cer(r, h)


def metricas_trainer(processor):
    tok = processor.tokenizer

    def f(pred):
        ids, lab = pred.predictions, pred.label_ids
        lab = np.where(lab == -100, tok.pad_token_id, lab)
        ids = np.where(ids == -100, tok.pad_token_id, ids)
        hyps = [normalizar(t) for t in tok.batch_decode(ids, skip_special_tokens=True)]
        refs = [normalizar(t) for t in tok.batch_decode(lab, skip_special_tokens=True)]
        w, c = wer_cer(refs, hyps)
        return {"wer": round(w, 2), "cer": round(c, 2)}
    return f


def wer_con_ic(pred, seed=0, n=1000):
    """WER corpus (no promedio por clip) e IC 95% por bootstrap sobre clips."""
    import jiwer
    p = pred[pred.text_norm.str.len() > 0]
    err, pal = [], []
    for a, b in zip(p.text_norm, p.prediccion):
        o = jiwer.process_words(a, b)
        err.append(o.substitutions + o.deletions + o.insertions)
        pal.append(o.substitutions + o.deletions + o.hits)
    err, pal = np.array(err), np.array(pal)
    rng = np.random.RandomState(seed)
    k = rng.randint(0, len(p), size=(n, len(p)))
    boot = 100 * err[k].sum(1) / pal[k].sum(1)
    return dict(wer=round(float(100 * err.sum() / pal.sum()), 2), ic95=[round(float(x), 2) for x in np.percentile(boot, [2.5, 97.5])],
                cer=round(100 * jiwer.cer(list(p.text_norm), list(p.prediccion)), 2), clips=int(len(p)),
                palabras=int(pal.sum()))


@torch.no_grad()
def transcribir(model, processor, df, raiz, device, batch=16, max_tokens=128, num_beams=1):
    """Transcribe df y devuelve una copia con la prediccion y el error por clip."""
    import jiwer
    model.eval()
    dtype = next(model.parameters()).dtype
    salida = []
    for i in range(0, len(df), batch):
        trozo = df.iloc[i:i + batch]
        audios = [sf.read(Path(raiz) / f, dtype="float32")[0] for f in trozo.file_name]
        x = processor.feature_extractor(audios, sampling_rate=SR, return_tensors="pt").input_features
        ids = model.generate(x.to(device, dtype), language="spanish", task="transcribe",
                             max_new_tokens=max_tokens, num_beams=num_beams)
        salida += processor.tokenizer.batch_decode(ids, skip_special_tokens=True)
    r = df[["id", "file_name", "source", "speaker_group", "duration_s", "text_norm"]].copy()
    r["prediccion"] = [normalizar(t) for t in salida]
    r["wer_clip"] = [round(100 * jiwer.wer(a, b), 1) if a else None for a, b in zip(r.text_norm, r.prediccion)]
    return r

In [ ]:
%%writefile /content/pseudo_core.py
# ---- pseudoetiquetado: se prueba localmente y se copia tal cual al notebook ----
import csv, os, zlib
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

SR = 16000


# ============================ CONFIANZA ============================
@torch.no_grad()
def transcribir_con_confianza(model, processor, rutas, device, max_tokens=128):
    """Greedy. Devuelve por clip: texto, log-prob promedio por token (incluye el fin de frase, como
    Whisper), numero de tokens y si toco el tope de tokens (señal de bucle)."""
    dtype = next(model.parameters()).dtype
    audios = [sf.read(r, dtype="float32")[0] for r in rutas]
    x = processor.feature_extractor(audios, sampling_rate=SR, return_tensors="pt").input_features
    out = model.generate(x.to(device, dtype), language="spanish", task="transcribe", max_new_tokens=max_tokens,
                         num_beams=1, do_sample=False, return_dict_in_generate=True, output_scores=True)
    lp = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True).float().cpu().numpy()
    gen = out.sequences[:, -lp.shape[1]:].cpu().numpy()          # tokens generados (sin el prompt forzado)
    eos = model.generation_config.eos_token_id
    eos = eos[0] if isinstance(eos, (list, tuple)) else eos
    res = []
    for fila_tok, fila_lp in zip(gen, lp):
        fin = np.where(fila_tok == eos)[0]
        n = int(fin[0]) + 1 if len(fin) else len(fila_tok)          # hasta el primer fin de frase, incluido
        res.append(dict(avg_logprob=round(float(np.mean(fila_lp[:n])), 4), n_tokens=n, toco_max=int(len(fin) == 0)))
    textos = processor.tokenizer.batch_decode(out.sequences, skip_special_tokens=True)
    for r, t in zip(res, textos):
        r["prediccion_cruda"] = t.strip()
    return res


# ============================ REPETICIONES ============================
def compression_ratio(t):
    """Criterio de Whisper para detectar bucles: un texto que se repite se comprime mucho (> 2.4)."""
    b = t.encode("utf-8")
    return round(len(b) / len(zlib.compress(b)), 3) if b else 0.0


def max_repeticion(t, n_max=4):
    """Mayor numero de veces que un n-grama (n = 1..n_max) se repite SEGUIDO. Devuelve (veces, n)."""
    w = t.split()
    mejor = (1, 1)
    for n in range(1, n_max + 1):
        i = 0
        while i + n <= len(w):
            k = 1
            while w[i + k * n: i + (k + 1) * n] == w[i: i + n]:
                k += 1
            if k > mejor[0]:
                mejor = (k, n)
            i += 1
    return mejor


def es_bucle(t, toco_max, comp_max=2.4, rep_1=4, rep_n=3):
    """True si el texto parece un bucle de Whisper. Una palabra repetida 2-3 veces es reduplicacion
    normal en shipibo ('ten ten'); 4 o mas seguidas, o una frase de 2-4 palabras 3 veces, no."""
    veces, n = max_repeticion(t)
    return bool(toco_max or compression_ratio(t) > comp_max or (n == 1 and veces >= rep_1) or (n > 1 and veces >= rep_n))


# ============================ LOTE REANUDABLE ============================
CAMPOS = ["id", "prediccion_cruda", "prediccion", "avg_logprob", "n_tokens", "toco_max"]


def transcribir_todo(df, raiz, cache, model, processor, device, normalizar, batch=16, max_tokens=128):
    hechos = set()
    if Path(cache).exists():
        with open(cache, encoding="utf-8", newline="") as f:
            hechos = {r["id"] for r in csv.DictReader(f) if r.get("id")}
    pend = df[~df.id.isin(hechos)]
    nuevo = not Path(cache).exists()
    from tqdm.auto import tqdm
    model.eval()
    with open(cache, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CAMPOS)
        if nuevo:
            w.writeheader()
        for i in tqdm(range(0, len(pend), batch), desc="transcribiendo"):
            t = pend.iloc[i:i + batch]
            res = transcribir_con_confianza(model, processor, [os.path.join(raiz, a) for a in t.file_name],
                                            device, max_tokens)
            for id_, r in zip(t.id, res):
                w.writerow(dict(id=id_, prediccion=normalizar(r["prediccion_cruda"]), **r))
            f.flush()
    return len(pend)

## 5. Datos y rango de caracteres por segundo del audio real

In [ ]:
import sys, importlib, json, re, shutil, zipfile
sys.path.insert(0, "/content")
import asr_core as C, pseudo_core as P
importlib.reload(C); importlib.reload(P)
import numpy as np, pandas as pd
from pathlib import Path

out = Path(SALIDA); out.mkdir(parents=True, exist_ok=True)


def id_de(v):
    v = v.strip()
    m = re.search(r"/d/([-\w]{20,})", v) or re.search(r"[?&]id=([-\w]{20,})", v)
    return m.group(1) if m else v


def abrir_zip(valor, ruta, que):
    if valor.strip():
        import gdown
        z = Path(gdown.download(id=id_de(valor), output="/content/", quiet=True) or "")
    else:
        z = Path(ruta)
    if not z.is_file():
        raise SystemExit(f"No encuentro {que} ({z}). Súbelo a Drive o pon su ID en la celda 1.")
    sha = C.sha256(z)
    destino = Path("/content/datos") / sha[:12]
    if not destino.exists():
        tmp = destino.with_name(destino.name + ".tmp"); shutil.rmtree(tmp, ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall(tmp)
        tmp.rename(destino)
    return next(destino.rglob("metadata.csv")).parent, sha


RAIZ_NP, sha_np = abrir_zip(NP_ID, DRIVE_NP, "no_pareado_v1.zip")
RAIZ_REAL, sha_real = abrir_zip(DATASET_ID, DRIVE_DATASET, "dataset_v1.0.zip")
np_meta = pd.read_csv(RAIZ_NP / "metadata.csv", keep_default_na=False)
real = pd.read_csv(RAIZ_REAL / "metadata.csv")
print(f"no pareado: {len(np_meta)} clips, {np_meta.duration_s.sum() / 60:.1f} min | "
      f"control transcrito: {(np_meta.en_control == 'transcrito').sum()}")

cps = lambda texto, dur: texto.str.replace(" ", "").str.len() / dur
real["cps"] = cps(real.text_norm, real.duration_s)
CPS_BAJO, CPS_ALTO = np.percentile(real.cps, [P_BAJO, P_ALTO])
ctrl = np_meta[np_meta.en_control == "transcrito"].copy()
ctrl_cps = cps(ctrl.text_norm_control, ctrl.duration_s)
print(f"caracteres/s del audio real pareado: p{P_BAJO} = {CPS_BAJO:.2f}, mediana = {real.cps.median():.2f}, p{P_ALTO} = {CPS_ALTO:.2f}")
print(f"control humano: mediana = {ctrl_cps.median():.2f}; fuera del rango: {100 * (~ctrl_cps.between(CPS_BAJO, CPS_ALTO)).mean():.1f}% "
      "(si esto es alto, el filtro 3 descartaría etiquetas correctas)")

## 6. Transcribir los 740 clips (reanudable)

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
FINAL = Path(ASR_DIR) / "modelo_final"
if not (FINAL / "config.json").exists():
    raise SystemExit(f"No existe {FINAL}: corre antes cerrar_asr_semilla.ipynb")
processor = WhisperProcessor.from_pretrained(FINAL)
asr = WhisperForConditionalGeneration.from_pretrained(FINAL)
asr.generation_config.forced_decoder_ids = None
device = torch.device("cuda")
asr = asr.to(device).half() if FP16 else asr.to(device)

n = P.transcribir_todo(np_meta, str(RAIZ_NP), out / "transcripciones.csv", asr, processor, device,
                       C.normalizar, BATCH, MAX_TOKENS)
print(f"transcritos ahora: {n}")
tr = pd.read_csv(out / "transcripciones.csv", keep_default_na=False).drop_duplicates("id", keep="last")
d = np_meta.merge(tr, on="id", how="left")
d["avg_logprob"] = pd.to_numeric(d.avg_logprob)
d["toco_max"] = pd.to_numeric(d.toco_max).astype(int)
d["compression_ratio"] = d.prediccion.map(P.compression_ratio)
d["rep_max"] = d.prediccion.map(lambda t: P.max_repeticion(t)[0])
d["cps"] = cps(d.prediccion, d.duration_s)
assert d.avg_logprob.notna().all(), "faltan clips por transcribir: vuelve a ejecutar esta celda"

## 7. ¿Qué tan bien pseudoetiqueta? Evaluación contra el control set

In [ ]:
import jiwer
ev = d[d.en_control == "transcrito"].copy()
ev = ev.rename(columns={"text_norm_control": "text_norm"})


def tildes_fuera(t):
    import unicodedata
    t = unicodedata.normalize("NFD", str(t))
    return unicodedata.normalize("NFC", "".join(ch for ch in t if ch not in "\u0301\u0300"))


def metricas(df):
    df2 = df.assign(text_norm=df.text_norm.map(tildes_fuera), prediccion=df.prediccion.map(tildes_fuera))
    return {"con_tildes": C.wer_con_ic(df), "sin_tildes": C.wer_con_ic(df2)}


ev["wer_clip"] = [round(100 * jiwer.wer(a, b), 1) if a else None for a, b in zip(ev.text_norm, ev.prediccion)]
ev[["id", "control_id", "source", "duration_s", "text_norm", "prediccion", "avg_logprob", "wer_clip"]].to_csv(
    out / "pred_control.csv", index=False)
EVAL = {"control (todo)": metricas(ev)}
for s, g in ev.groupby("source"):
    EVAL[s] = metricas(g)
filas = [{"conjunto": k, "clips": v["con_tildes"]["clips"], "palabras": v["con_tildes"]["palabras"],
          "WER": f"{v['con_tildes']['wer']} [{v['con_tildes']['ic95'][0]}–{v['con_tildes']['ic95'][1]}]",
          "CER": v["con_tildes"]["cer"], "WER sin tildes": v["sin_tildes"]["wer"], "CER sin tildes": v["sin_tildes"]["cer"]}
         for k, v in EVAL.items()]
print("ASR semilla en audio NO pareado (contra la transcripción humana)\n")
print(pd.DataFrame(filas).set_index("conjunto").to_string())

## 8. Filtros

In [ ]:
def filtrar(df, umbral_conf):
    """Aplica los filtros en orden; devuelve la columna 'descarte' (vacío = aceptado)."""
    desc = pd.Series("", index=df.index)
    reglas = [
        ("1 vacío", df.prediccion.str.strip() == ""),
        ("2 repeticiones", [P.es_bucle(t, m, COMP_MAX, REP_1, REP_N) for t, m in zip(df.prediccion, df.toco_max)]),
        ("3 caracteres/s", ~df.cps.between(CPS_BAJO, CPS_ALTO)),
        ("4 confianza", df.avg_logprob < umbral_conf),
    ]
    for nombre, m in reglas:
        m = pd.Series(m, index=df.index).astype(bool)
        desc[(desc == "") & m] = nombre
    return desc


pool = d[d.en_control != "transcrito"].copy()          # lo que se pseudoetiqueta (sin el control)
pre = filtrar(pool, -np.inf)                             # filtros 1-3, sin confianza
if MODO_CONFIANZA == "top":
    UMBRAL = float(np.percentile(pool.loc[pre == "", "avg_logprob"], 100 - TOP_PCT))
else:
    UMBRAL = UMBRAL_LOGPROB
print(f"umbral de confianza (log-prob promedio): {UMBRAL:.3f}  [{MODO_CONFIANZA}]")
pool["descarte"] = filtrar(pool, UMBRAL)
ev["descarte"] = filtrar(ev, UMBRAL)                     # mismos filtros y mismo umbral sobre el control

# --- calidad esperada de las etiquetas aceptadas
acept_c, rech_c = ev[ev.descarte == ""], ev[ev.descarte != ""]
CAL = {"aceptados": metricas(acept_c) if len(acept_c) else None,
       "rechazados": metricas(rech_c) if len(rech_c) else None}
for k, v in CAL.items():
    if v:
        print(f"control {k:10s}: {v['con_tildes']['clips']:3d} clips | WER {v['con_tildes']['wer']} "
              f"[{v['con_tildes']['ic95'][0]}–{v['con_tildes']['ic95'][1]}] | WER sin tildes {v['sin_tildes']['wer']}")
print("\n=> el WER de 'aceptados' es el error esperado de las etiquetas en metadata_pseudo.csv")

# --- curva: WER de las etiquetas que se quedan según el % que se conserva
import matplotlib.pyplot as plt
curva = []
for pct in range(10, 101, 10):
    u = float(np.percentile(pool.loc[pre == "", "avg_logprob"], 100 - pct)) if pct < 100 else -np.inf
    k = ev[filtrar(ev, u) == ""]
    if len(k) >= 5:
        curva.append(dict(pct_conservado=pct, umbral=round(u, 3), clips_control=len(k),
                          wer=C.wer_con_ic(k)["wer"],
                          minutos_pseudo=round(pool.loc[filtrar(pool, u) == "", "duration_s"].sum() / 60, 1)))
curva = pd.DataFrame(curva)
curva.to_csv(out / "curva_confianza.csv", index=False)
print("\n" + curva.to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(curva.pct_conservado, curva.wer, marker="o")
ax.set_xlabel("% del pool que se conserva (por confianza)"); ax.set_ylabel("WER de las etiquetas que quedan (%)")
ax.axvline(TOP_PCT if MODO_CONFIANZA == "top" else 100, color="gray", ls="--")
for r in curva.itertuples():
    ax.annotate(f"{r.minutos_pseudo} min", (r.pct_conservado, r.wer), fontsize=7, textcoords="offset points", xytext=(0, 6))
plt.tight_layout(); plt.savefig(out / "curva_confianza.png", dpi=120); plt.show()

## 9. `metadata_pseudo.csv` y tabla resumen

In [ ]:
m = lambda df: round(df.duration_s.sum() / 60, 2)
filas = [dict(etapa="total no pareado", clips=len(d), minutos=m(d)),
         dict(etapa="control set (evaluación, no se usa)", clips=-len(ev), minutos=-m(ev)),
         dict(etapa="pool a pseudoetiquetar", clips=len(pool), minutos=m(pool))]
for nombre in ["1 vacío", "2 repeticiones", "3 caracteres/s", "4 confianza"]:
    x = pool[pool.descarte == nombre]
    filas.append(dict(etapa=f"rechazados: {nombre}", clips=-len(x), minutos=-m(x)))
acept = pool[pool.descarte == ""]
filas.append(dict(etapa="ACEPTADOS", clips=len(acept), minutos=m(acept)))
resumen = pd.DataFrame(filas)
resumen.to_csv(out / "resumen_pseudo.csv", index=False)
print(resumen.to_string(index=False))

meta_out = pd.DataFrame({
    "id": acept.id, "file_name": acept.file_name, "transcription": acept.prediccion, "text_norm": acept.prediccion,
    "source": acept.source, "subsource": acept.subsource, "speaker_group": acept.speaker_group,
    "duration_s": acept.duration_s, "avg_logprob": acept.avg_logprob, "cps": acept.cps.round(2),
    "compression_ratio": acept.compression_ratio, "pseudo_etiqueta": True})
meta_out.to_csv(out / "metadata_pseudo.csv", index=False)
pd.concat([pool, ev]).to_csv(out / "detalle_pseudo.csv", index=False)

cfg = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool))}
cfg.update(cps_rango=[float(CPS_BAJO), float(CPS_ALTO)], umbral_confianza_usado=UMBRAL,
           sha256_no_pareado=sha_np, sha256_dataset=sha_real, sha256_modelo=C.sha256(FINAL / "model.safetensors"))
(out / "config.json").write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
(out / "eval_control.json").write_text(json.dumps({"semilla_en_control": EVAL, "calidad_por_filtro": CAL,
                                                   "umbral": UMBRAL}, indent=2, ensure_ascii=False))
print(f"\n{out / 'metadata_pseudo.csv'}: {len(meta_out)} clips, {m(acept)} min "
      f"(rutas relativas a no_pareado_v1/)")